# Cross-Validation and Hyperparameter Tuning

In this notebook, we learn cross-validation for evaluating model generalization performance and methods for finding optimal hyperparameters.

## Table of Contents
1. Cross-Validation
   - K-Fold Cross-Validation
   - Stratified K-Fold
   - Leave-One-Out CV
   - Time Series Split
2. Hyperparameter Tuning
   - Grid Search
   - Randomized Search
3. Advanced Techniques
   - Nested Cross-Validation
   - Learning Curves

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, load_breast_cancer, load_diabetes
from sklearn.model_selection import (
    cross_val_score, cross_validate,
    KFold, StratifiedKFold, LeaveOneOut, ShuffleSplit,
    TimeSeriesSplit, RepeatedKFold,
    GridSearchCV, RandomizedSearchCV,
    learning_curve, validation_curve
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, f1_score, accuracy_score
from scipy.stats import uniform, randint

# Visualization settings
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

## 1. Cross-Validation

### 1.1 K-Fold Cross-Validation

In [ ]:
# Load data
iris = load_iris()
X, y = iris.data, iris.target

# Create model
model = LogisticRegression(max_iter=1000, random_state=42)

# Why: A single train/test split is noisy — performance depends heavily on which
# samples end up in test. K-Fold CV rotates through K splits, averaging scores
# to give a more stable and reliable estimate of generalization performance.
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print("=== K-Fold Cross-Validation (K=5) ===")
print(f"Score per fold: {scores}")
print(f"Mean accuracy: {scores.mean():.4f}")
print(f"Standard deviation: {scores.std():.4f}")
print(f"95% confidence interval: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")

In [ ]:
# Manual K-Fold implementation for understanding
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("\n=== K-Fold Split Visualization ===")
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    # Data splitting
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # Model training and evaluation
    model_fold = LogisticRegression(max_iter=1000, random_state=42)
    model_fold.fit(X_train, y_train)
    score = model_fold.score(X_val, y_val)
    fold_scores.append(score)
    
    print(f"Fold {fold}: Train={len(train_idx)}, Val={len(val_idx)}, Accuracy={score:.4f}")

print(f"\nMean accuracy: {np.mean(fold_scores):.4f}")

In [ ]:
# Performance comparison by K value
k_values = [3, 5, 10, 15, 20]
mean_scores = []
std_scores = []

for k in k_values:
    scores = cross_val_score(model, X, y, cv=k, scoring='accuracy')
    mean_scores.append(scores.mean())
    std_scores.append(scores.std())

# Visualization
plt.figure(figsize=(10, 6))
plt.errorbar(k_values, mean_scores, yerr=std_scores, marker='o', 
             capsize=5, capthick=2, linewidth=2)
plt.xlabel('K (Number of Folds)', fontsize=12)
plt.ylabel('Mean Accuracy', fontsize=12)
plt.title('K-Fold CV Performance vs K Value', fontsize=14, pad=20)
plt.grid(True, alpha=0.3)
plt.show()

print("\n=== Performance by K Value ===")
for k, mean, std in zip(k_values, mean_scores, std_scores):
    print(f"K={k:2d}: {mean:.4f} (+/- {std:.4f})")

### 1.2 Stratified K-Fold

In [ ]:
# Check class distribution
unique, counts = np.unique(y, return_counts=True)
print("Full dataset class distribution:")
for cls, cnt in zip(unique, counts):
    print(f"  Class {cls}: {cnt} ({cnt/len(y)*100:.1f}%)")

# Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n=== Stratified K-Fold Class Distribution per Fold ===")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    train_classes = np.bincount(y[train_idx])
    val_classes = np.bincount(y[val_idx])
    
    print(f"Fold {fold}:")
    print(f"  Train: {train_classes} ({train_classes/train_classes.sum()*100})")
    print(f"  Val:   {val_classes} ({val_classes/val_classes.sum()*100})")

In [ ]:
# K-Fold vs Stratified K-Fold performance comparison
kf = KFold(n_splits=5, shuffle=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores_kf = cross_val_score(model, X, y, cv=kf, scoring='accuracy')
scores_skf = cross_val_score(model, X, y, cv=skf, scoring='accuracy')

print("=== K-Fold vs Stratified K-Fold ===")
print(f"K-Fold:           {scores_kf.mean():.4f} (+/- {scores_kf.std():.4f})")
print(f"Stratified K-Fold: {scores_skf.mean():.4f} (+/- {scores_skf.std():.4f})")
print("\nStratified K-Fold is more stable for imbalanced datasets.")

### 1.3 Various Cross-Validation Methods

In [ ]:
# Leave-One-Out (LOO)
loo = LeaveOneOut()
print(f"Leave-One-Out number of splits: {loo.get_n_splits(X)} (same as sample count)")
print("LOO is useful for small datasets but has high computational cost.\n")

# Shuffle Split (Random Split)
ss = ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
scores_ss = cross_val_score(model, X, y, cv=ss, scoring='accuracy')
print(f"Shuffle Split mean: {scores_ss.mean():.4f} (+/- {scores_ss.std():.4f})")
print("Each split is independently randomly sampled.\n")

# Repeated K-Fold
rkf = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)
scores_rkf = cross_val_score(model, X, y, cv=rkf, scoring='accuracy')
print(f"Repeated K-Fold mean: {scores_rkf.mean():.4f} (+/- {scores_rkf.std():.4f})")
print(f"Total number of splits: {len(scores_rkf)} (5 folds x 10 repeats = 50)")
print("Multiple repetitions provide a more stable estimate.")

### 1.4 Time Series Cross-Validation (Time Series Split)

In [ ]:
# Cross-validation for time series data
tscv = TimeSeriesSplit(n_splits=5)

print("=== Time Series Split ===")
print("Time series data must maintain past-to-future ordering.\n")

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
    print(f"Fold {fold}:")
    print(f"  Train: [{train_idx[0]:3d}:{train_idx[-1]:3d}] ({len(train_idx)} samples)")
    print(f"  Test:  [{test_idx[0]:3d}:{test_idx[-1]:3d}] ({len(test_idx)} samples)")

In [ ]:
# Time Series Split visualization
fig, ax = plt.subplots(figsize=(12, 6))

for i, (train, test) in enumerate(tscv.split(X)):
    # Train set
    ax.barh(i, len(train), left=train[0], height=0.4, 
            align='center', color='blue', alpha=0.6, label='Train' if i == 0 else '')
    # Test set
    ax.barh(i, len(test), left=test[0], height=0.4, 
            align='center', color='red', alpha=0.6, label='Test' if i == 0 else '')

ax.set_yticks(range(tscv.n_splits))
ax.set_yticklabels([f'Fold {i+1}' for i in range(tscv.n_splits)])
ax.set_xlabel('Sample Index', fontsize=12)
ax.set_title('Time Series Split Visualization', fontsize=14, pad=20)
ax.legend(loc='upper left', fontsize=11)
plt.tight_layout()
plt.show()

## 2. cross_val_score vs cross_validate

In [ ]:
# cross_validate: evaluate multiple metrics simultaneously
scoring = ['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted']

cv_results = cross_validate(
    model, X, y,
    cv=5,
    scoring=scoring,
    return_train_score=True
)

print("=== cross_validate Results ===")
for metric in scoring:
    train_key = f'train_{metric}'
    test_key = f'test_{metric}'
    print(f"\n{metric}:")
    print(f"  Train: {cv_results[train_key].mean():.4f} (+/- {cv_results[train_key].std():.4f})")
    print(f"  Test:  {cv_results[test_key].mean():.4f} (+/- {cv_results[test_key].std():.4f})")

print(f"\nMean fit time: {cv_results['fit_time'].mean():.4f} sec")
print(f"Mean score time: {cv_results['score_time'].mean():.4f} sec")

In [ ]:
# Results visualization
metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Train': [cv_results[f'train_{m}'].mean() for m in scoring],
    'Test': [cv_results[f'test_{m}'].mean() for m in scoring]
})

x = np.arange(len(metrics_df))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, metrics_df['Train'], width, label='Train', alpha=0.8)
bars2 = ax.bar(x + width/2, metrics_df['Test'], width, label='Test', alpha=0.8)

ax.set_xlabel('Metrics', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Train vs Test Scores (5-Fold CV)', fontsize=14, pad=20)
ax.set_xticks(x)
ax.set_xticklabels(metrics_df['Metric'])
ax.legend(fontsize=11)
ax.set_ylim([0.9, 1.0])
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 3. Hyperparameter Tuning

### 3.1 Grid Search

In [ ]:
# Why: Scaling BEFORE GridSearchCV would cause data leakage — test fold statistics
# would contaminate training. Here we pre-scale for simplicity, but the Pipeline
# approach in cell 31 is the correct production pattern.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cancer)

# Hyperparameter grid
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [1, 0.1, 0.01, 0.001],
    'kernel': ['rbf', 'linear']
}

print("=== Grid Search ===")
print(f"Number of parameter combinations: {len(param_grid['C']) * len(param_grid['gamma']) * len(param_grid['kernel'])}")
print(f"CV Folds: 5")
print(f"Total fits: {32 * 5} = 160\n")

In [ ]:
# Run Grid Search
grid_search = GridSearchCV(
    SVC(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1  # Use all CPUs
)

grid_search.fit(X_scaled, y_cancer)

print("\n=== Grid Search Results ===")
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best score: {grid_search.best_score_:.4f}")
print(f"Best model: {grid_search.best_estimator_}")

In [ ]:
# View all results
results_df = pd.DataFrame(grid_search.cv_results_)

# Top 10 combinations
top_results = results_df.nsmallest(10, 'rank_test_score')[[
    'params', 'mean_test_score', 'std_test_score', 'rank_test_score'
]]

print("\n=== Top 10 Parameter Combinations ===")
for idx, row in top_results.iterrows():
    print(f"Rank {int(row['rank_test_score'])}: {row['params']}")
    print(f"  Score: {row['mean_test_score']:.4f} (+/- {row['std_test_score']:.4f})\n")

In [ ]:
# Grid Search results heatmap (C vs gamma, rbf kernel)
rbf_results = results_df[results_df['param_kernel'] == 'rbf']

# Create pivot table
pivot_table = rbf_results.pivot_table(
    values='mean_test_score',
    index='param_gamma',
    columns='param_C'
)

plt.figure(figsize=(10, 8))
sns.heatmap(pivot_table, annot=True, fmt='.4f', cmap='YlGnBu', 
            cbar_kws={'label': 'Accuracy'})
plt.title('Grid Search Results (RBF Kernel): C vs Gamma', fontsize=14, pad=20)
plt.xlabel('C (Regularization Parameter)', fontsize=12)
plt.ylabel('Gamma', fontsize=12)
plt.tight_layout()
plt.show()

### 3.2 Randomized Search

In [ ]:
# Define hyperparameter distributions
param_distributions = {
    'C': uniform(0.1, 100),  # 0.1 ~ 100.1 uniform distribution
    'gamma': uniform(0.001, 1),  # 0.001 ~ 1.001 uniform distribution
    'kernel': ['rbf', 'linear', 'poly']
}

# Run Randomized Search
random_search = RandomizedSearchCV(
    SVC(random_state=42),
    param_distributions,
    n_iter=50,  # Try 50 combinations
    cv=5,
    scoring='accuracy',
    random_state=42,
    verbose=1,
    n_jobs=-1
)

random_search.fit(X_scaled, y_cancer)

print("\n=== Randomized Search Results ===")
print(f"Best parameters: {random_search.best_params_}")
print(f"Best score: {random_search.best_score_:.4f}")

In [ ]:
# Grid Search vs Randomized Search comparison
comparison_df = pd.DataFrame({
    'Method': ['Grid Search', 'Randomized Search'],
    'Best Score': [grid_search.best_score_, random_search.best_score_],
    'N Iterations': [len(grid_search.cv_results_['params']), 
                     len(random_search.cv_results_['params'])]
})

print("\n=== Grid Search vs Randomized Search ===")
print(comparison_df.to_string(index=False))
print("\nRandomized Search:")
print("  - Pros: Computationally efficient, can explore continuous distributions")
print("  - Cons: No guarantee of finding the optimal solution")
print("\nGrid Search:")
print("  - Pros: Exhaustive search, guaranteed optimum (within grid)")
print("  - Cons: Number of combinations grows exponentially")

### 3.3 Random Forest Hyperparameter Tuning

In [ ]:
# Random Forest parameter grid
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_param_grid,
    cv=5,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)

rf_grid_search.fit(X_cancer, y_cancer)

print("\n=== Random Forest Grid Search Results ===")
print(f"Best parameters: {rf_grid_search.best_params_}")
print(f"Best score: {rf_grid_search.best_score_:.4f}")

In [ ]:
# Feature Importance visualization
best_rf = rf_grid_search.best_estimator_
feature_importance = pd.DataFrame({
    'feature': cancer.feature_names,
    'importance': best_rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
plt.barh(feature_importance['feature'][:15], feature_importance['importance'][:15])
plt.xlabel('Importance', fontsize=12)
plt.title('Top 15 Feature Importances (Optimized Random Forest)', fontsize=14, pad=20)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 5 Features:")
print(feature_importance.head().to_string(index=False))

## 4. Nested Cross-Validation

In [ ]:
# Why: Nested CV separates model selection (inner loop) from performance estimation
# (outer loop). Without nesting, the same data used to pick the best hyperparams
# also estimates performance, leading to optimistic bias (information leakage).

# Inner CV (hyperparameter tuning)
param_grid_nested = {'C': [0.1, 1, 10], 'gamma': [0.1, 0.01]}
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
grid_search_nested = GridSearchCV(
    SVC(kernel='rbf', random_state=42), 
    param_grid_nested, 
    cv=inner_cv, 
    scoring='accuracy'
)

# Outer CV (model evaluation)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
nested_scores = cross_val_score(
    grid_search_nested, 
    X_scaled, 
    y_cancer, 
    cv=outer_cv, 
    scoring='accuracy'
)

print("=== Nested Cross-Validation Results ===")
print(f"Score per outer fold: {nested_scores}")
print(f"Mean score: {nested_scores.mean():.4f} (+/- {nested_scores.std():.4f})")

# Comparison: Standard CV vs Nested CV
grid_search_nested.fit(X_scaled, y_cancer)
print(f"\nStandard CV best score: {grid_search_nested.best_score_:.4f}")
print(f"Nested CV mean score: {nested_scores.mean():.4f}")
print("\nNested CV provides a more realistic estimate of generalization performance.")
print("Standard CV may overestimate (data leakage).")

## 5. Using with Pipeline

In [ ]:
# Why: A Pipeline ensures that scaling is applied INSIDE each CV fold, preventing
# data leakage. Without it, test-fold statistics leak into training via the scaler.
# Parameter names use 'step__param' syntax to identify which step to tune.
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(random_state=42))
])

# Parameter names: step__parameter
param_grid_pipeline = {
    'svm__C': [0.1, 1, 10],
    'svm__gamma': [0.1, 0.01, 0.001],
    'svm__kernel': ['rbf', 'linear']
}

grid_search_pipeline = GridSearchCV(
    pipeline, 
    param_grid_pipeline, 
    cv=5, 
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)

# Use unscaled data (pipeline handles scaling)
grid_search_pipeline.fit(X_cancer, y_cancer)

print("\n=== Pipeline Grid Search Results ===")
print(f"Best parameters: {grid_search_pipeline.best_params_}")
print(f"Best score: {grid_search_pipeline.best_score_:.4f}")
print("\nAdvantages of using Pipeline:")
print("  - Automatically includes preprocessing steps")
print("  - Prevents data leakage in CV")
print("  - Code conciseness")

## 6. Learning Curves

In [ ]:
# Compute learning curves
train_sizes, train_scores, val_scores = learning_curve(
    grid_search_pipeline.best_estimator_,
    X_cancer, y_cancer,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

# Mean and standard deviation
train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

# Learning curve visualization
plt.figure(figsize=(10, 6))
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, 
                 alpha=0.2, color='blue')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, 
                 alpha=0.2, color='orange')
plt.plot(train_sizes, train_mean, 'o-', color='blue', linewidth=2, 
         label='Training Score')
plt.plot(train_sizes, val_mean, 'o-', color='orange', linewidth=2, 
         label='Validation Score')
plt.xlabel('Training Set Size', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Learning Curve', fontsize=14, pad=20)
plt.legend(loc='best', fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

print("Learning curve interpretation:")
print("  - Both curves low -> Underfitting")
print("  - Training curve high, validation curve low -> Overfitting")
print("  - Both curves converge -> Good fit")

## 7. Validation Curve

In [ ]:
# Validation curve for C parameter
param_range = np.logspace(-4, 2, 10)

train_scores_val, test_scores_val = validation_curve(
    SVC(kernel='rbf', gamma=0.01, random_state=42),
    X_scaled, y_cancer,
    param_name='C',
    param_range=param_range,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

train_mean_val = train_scores_val.mean(axis=1)
train_std_val = train_scores_val.std(axis=1)
test_mean_val = test_scores_val.mean(axis=1)
test_std_val = test_scores_val.std(axis=1)

# Validation curve visualization
plt.figure(figsize=(10, 6))
plt.semilogx(param_range, train_mean_val, 'o-', color='blue', linewidth=2, 
             label='Training Score')
plt.semilogx(param_range, test_mean_val, 'o-', color='orange', linewidth=2, 
             label='Validation Score')
plt.fill_between(param_range, train_mean_val - train_std_val, 
                 train_mean_val + train_std_val, alpha=0.2, color='blue')
plt.fill_between(param_range, test_mean_val - test_std_val, 
                 test_mean_val + test_std_val, alpha=0.2, color='orange')
plt.xlabel('C (Regularization Parameter)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Validation Curve (SVM RBF)', fontsize=14, pad=20)
plt.legend(loc='best', fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

print("Validation curve interpretation:")
print("  - Left (small C): Underfitting (strong regularization)")
print("  - Middle: Appropriate complexity")
print("  - Right (large C): Potential overfitting (weak regularization)")

## 8. Custom Scoring Function

In [ ]:
# Custom scoring function
def custom_f1_score(y_true, y_pred):
    """Weighted average F1-score"""
    return f1_score(y_true, y_pred, average='weighted')

custom_scorer = make_scorer(custom_f1_score)

# Use custom scorer
scores_custom = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=42), 
    X, y, 
    cv=5, 
    scoring=custom_scorer
)

print("=== Custom Scoring Function ===")
print(f"Custom F1-score: {scores_custom.mean():.4f} (+/- {scores_custom.std():.4f})")

# Built-in scoring functions
print("\nBuilt-in scoring functions:")
print("  Classification: 'accuracy', 'precision', 'recall', 'f1', 'roc_auc'")
print("  Regression: 'r2', 'neg_mean_squared_error', 'neg_mean_absolute_error'")

## 9. Saving and Loading Results

In [ ]:
import joblib
import json

# Save the best model
best_model = grid_search_pipeline.best_estimator_
joblib.dump(best_model, 'best_model.pkl')
print("Best model saved to 'best_model.pkl'.")

# Save results
results = {
    'best_params': grid_search_pipeline.best_params_,
    'best_score': grid_search_pipeline.best_score_,
    'cv_results': {
        k: v.tolist() if isinstance(v, np.ndarray) else v
        for k, v in grid_search_pipeline.cv_results_.items()
        if k in ['params', 'mean_test_score', 'std_test_score', 'rank_test_score']
    }
}

with open('tuning_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Tuning results saved to 'tuning_results.json'.")

# Load model
loaded_model = joblib.load('best_model.pkl')
print(f"\nLoaded model: {loaded_model}")

## Summary

### Choosing a Cross-Validation Method

| Technique | Use Case | Characteristics |
|-----------|----------|-----------------|
| K-Fold | General evaluation | Splits data into K parts |
| Stratified K-Fold | Imbalanced data | Preserves class proportions |
| Time Series Split | Time series data | Maintains temporal order |
| Leave-One-Out | Small datasets | High computational cost |

### Hyperparameter Tuning Methods

| Method | Pros | Cons | When to Use |
|--------|------|------|-------------|
| Grid Search | Exhaustive search | High computational cost | Few parameters, clear range |
| Randomized Search | Efficient | No optimality guarantee | Many parameters, uncertain range |
| Nested CV | High reliability | Very high computational cost | Research, benchmarks |

### Practical Tips

1. **Small datasets**: Stratified K-Fold (k=5 or 10)
2. **Large datasets**: Stratified K-Fold (k=3) or single train/test split
3. **Time series**: Time Series Split
4. **Parameter search**: Grid Search (narrow range) -> Randomized Search (wide range)
5. **Pipeline**: Include preprocessing to prevent data leakage